# Introduccion a Hugging Face: Biblioteca de modelos de IA

Hugging Face es como el GitHub de los modelos de inteligencia artificial. Aqui encontraras miles de modelos preentrenados listos para usar, sin necesidad de entrenarlos desde cero. En este notebook aprenderemos a buscar, descargar y usar modelos de manera sencilla.

## Configuracion e Imports

Importamos las librerias necesarias. **Transformers** es la libreria principal de Hugging Face que nos permite usar modelos de manera muy sencilla.

In [ ]:
import torch
import transformers
from transformers import AutoImageProcessor, AutoModelForImageClassification
from huggingface_hub import snapshot_download
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import numpy as np
import requests
from io import BytesIO
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch version: {torch.__version__}, usando: {device}")
print(f"transformers version: {transformers.__version__}")

## Que es Hugging Face Hub?

Hugging Face Hub es una plataforma donde la comunidad sube modelos preentrenados. Podemos:

- **Buscar modelos** por tarea (clasificacion, deteccion, segmentacion, etc.)
- **Descargar modelos** de manera automatica
- **Ver licencias** para saber si podemos usar el modelo comercialmente
- **Leer documentacion** y ejemplos de uso

**Pagina web**: https://huggingface.co

**Como buscar un modelo:**
1. Entra en https://huggingface.co/models
2. Usa los filtros de la izquierda para seleccionar la tarea (ej: Image Classification, Object Detection)
3. Lee la tarjeta del modelo (Model Card) para entender que hace


## Metodo 1: Usar un modelo directamente (Recomendado)

La forma mas sencilla es usar `from_pretrained()`. Esto descarga el modelo automaticamente la primera vez y lo cachea para usos futuros.

Vamos a probar con un modelo de clasificacion de imagenes: **google/vit-base-patch16-224**

Este modelo es un Vision Transformer (ViT) entrenado en ImageNet con 1000 categorias.

In [ ]:
# Cargar modelo y procesador directamente desde HF Hub
model_name = "google/vit-base-patch16-224"

processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(model_name).to(device)

# Entrenado en Imagenet (más clases que COCO)
print(f"Modelo cargado: {model_name}")
print(f"Numero de clases: {model.config.num_labels}")
print(f"Clases: {model.config.id2label}")

## Probando el modelo con una imagen

Vamos a clasificar una imagen de ejemplo. El modelo nos dira que objeto detecta y con que confianza.

In [ ]:
# Cargar imagen de ejemplo desde GitHub
url = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/cars.jpg"
# url = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/dog2.jpg"

image = Image.open(BytesIO(requests.get(url).content)).convert("RGB")

# Procesar imagen y hacer prediccion
inputs = processor(images=image, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)
    
logits = outputs.logits


# Obtener top 5 predicciones
probs = torch.nn.functional.softmax(logits, dim=-1)[0]

top3_prob, top3_idx = torch.topk(probs, 3)

print(top3_prob)

# Mostrar resultados
print("Top 3 predicciones:\n")
for prob, idx in zip(top3_prob, top3_idx):
    label = model.config.id2label[idx.item()]
    print(f"{label}: {prob.item()*100:.2f}%")

# Visualizar imagen
plt.imshow(image)
plt.title(f"Prediccion: {model.config.id2label[top3_idx[0].item()]} ({top3_prob[0].item()*100:.1f}%)")
plt.axis("off")
plt.tight_layout()
plt.show()

## Metodo 2: Descargar modelo para usar offline

Si quieres descargar el modelo completo para usarlo sin conexion, puedes usar `snapshot_download()`. Esto es util si:

- Vas a usar el modelo muchas veces
- Quieres evitar descargas repetidas
- No tendras conexion a internet

**Nota**: Esto puede ocupar varios GB dependiendo del modelo.

In [ ]:
# Detectar si estamos en Colab o local
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    HF_MODELS_DIR = Path("/content/drive/MyDrive/hf_models")
else:
    HF_MODELS_DIR = Path("./hf_models")  # Carpeta local

HF_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Descargar modelo completo
model_name = "google/vit-base-patch16-224"
local_dir = HF_MODELS_DIR / model_name

print(f"Descargando modelo a: {local_dir}")
snapshot_download(
    repo_id=model_name,
    local_dir=str(local_dir),
    local_dir_use_symlinks=False,
    resume_download=True
)
print("Descarga completada!")

## Inspeccionando archivos del modelo

Un modelo de Hugging Face suele contener estos archivos:

- **config.json**: Configuracion del modelo (arquitectura, numero de capas, etc.)
- **model.safetensors** o **pytorch_model.bin**: Pesos del modelo
- **preprocessor_config.json**: Configuracion del preprocesamiento de imagenes
- **README.md**: Documentacion del modelo

Vamos a listar los archivos descargados:

In [ ]:
# Listar archivos del modelo
if local_dir.exists():
    print(f"Archivos en {local_dir}:\n")
    for file in sorted(local_dir.rglob("*")):
        if file.is_file():
            size_mb = file.stat().st_size / (1024**2)
            print(f"  {file.name} ({size_mb:.2f} MB)")
else:
    print("Ejecuta la celda anterior primero para descargar el modelo")

## Cargar modelo desde ruta local

Una vez descargado, podemos cargarlo desde la ruta local en lugar de descargarlo cada vez:

In [ ]:
# Cargar desde ruta local
if local_dir.exists():
    processor_local = AutoImageProcessor.from_pretrained(str(local_dir))
    model_local = AutoModelForImageClassification.from_pretrained(str(local_dir)).to(device)
    print("Modelo cargado desde ruta local")
else:
    print("Usa el Metodo 1 para cargar directamente sin descargar")

## Licencias en Hugging Face

Es importante revisar la licencia antes de usar un modelo, especialmente para uso comercial.

**Licencias comunes:**

- **Apache 2.0 / MIT**: Uso libre, incluso comercial
- **CreativeML Open RAIL-M**: Uso libre con restricciones eticas
- **GPL**: Codigo abierto, pero tu proyecto debe ser open source tambien
- **Research only**: Solo para investigacion, NO comercial

La licencia suele aparecer en la pagina del modelo en HF Hub, en la seccion de "Model Card".

## Ejercicio: Busca y prueba otro modelo

Ahora te toca a ti! Vamos a probar con otro modelo:

1. Ve a https://huggingface.co/models
2. Filtra por tarea: **Image Classification**
3. Encuentra un modelo que te interese (ej: uno entrenado en otra categoria de imagenes)
4. Copia el nombre del modelo (aparece como `usuario/nombre-modelo`)
5. Modifica el codigo de abajo y pruebalo!

**Sugerencias:**
- `microsoft/resnet-50`: Modelo clasico ResNet
- `facebook/convnext-tiny-224`: Arquitectura moderna
- `timm/mobilenetv3_large_100.ra_in1k`: Modelo ligero para dispositivos moviles

In [ ]:
# EJERCICIO: Cambia el nombre del modelo y pruebalo
tu_modelo = ""  # <- Cambia esto por el modelo que elijas

# Cargar modelo
processor_ej = AutoImageProcessor.from_pretrained(tu_modelo)
model_ej = AutoModelForImageClassification.from_pretrained(tu_modelo).to(device)

# Probar con imagen
inputs_ej = processor_ej(images=image, return_tensors="pt").to(device)

with torch.no_grad():
    outputs_ej = model_ej(**inputs_ej)
    probs_ej = torch.nn.functional.softmax(outputs_ej.logits, dim=-1)[0]
    top1_prob, top1_idx = torch.topk(probs_ej, 1)

# Mostrar resultado
label_ej = model_ej.config.id2label[top1_idx[0].item()]
print(f"Prediccion con {tu_modelo}:")
print(f"{label_ej}: {top1_prob[0].item()*100:.2f}%")

plt.imshow(image)
plt.title(f"{label_ej} ({top1_prob[0].item()*100:.1f}%)")
plt.axis("off")
plt.show()

## Ejercicio Extra (Comodin): Detectar objetos con un modelo de OD

Si has terminado rapido o quieres practicar mas en casa, prueba a buscar un modelo de **Object Detection** en HF Hub y usarlo.

**Pista**: Busca modelos con nombres como `facebook/detr-resnet-50`

El proceso es similar pero la salida sera diferente (bounding boxes en lugar de etiquetas).

In [ ]:
# EJERCICIO EXTRA: Intenta usar un modelo de Object Detection como `facebook/detr-resnet-50`
# Pista: from transformers import AutoImageProcessor, AutoModelForObjectDetection

In [ ]:
# --- AÑADIR: Visualizar detecciones con cv2 ---
# img_array = np.array(image)
# img_result = img_array.copy()

# for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
#     x1, y1, x2, y2 = map(int, box.tolist())
#     class_name = model.config.id2label[label.item()]
    
#     # Dibujar box
#     cv2.rectangle(img_result, (x1, y1), (x2, y2), (0, 0, 255), 3)
#     text = f"{class_name} {score:.2f}"
#     cv2.putText(img_result, text, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

# # Visualizar
# plt.imshow(img_result)
# plt.title(f"Detecciones Object Detection (threshold=0.9)")
# plt.axis("off")
# plt.tight_layout()
# plt.show()

## Resumen

En este notebook hemos aprendido:

✅ Que es Hugging Face Hub y como buscar modelos  
✅ Dos formas de descargar modelos: directa y offline  
✅ Como usar `from_pretrained()` para cargar modelos  
✅ Inspeccionar archivos de un modelo  
✅ Importancia de revisar las licencias  
✅ Hacer inferencia con un modelo de clasificacion y otro de OD 